# 001 — Boosting Showdown: CatBoost vs XGBoost vs LightGBM

A comprehensive comparison of the three dominant gradient boosting frameworks
for binary classification. We train each model with Optuna hyperparameter
tuning, compare them on metrics and threshold analysis, and examine what they
say about feature importance and SHAP values.

**Lifecycle stage:** seedling (model-garden)

All code is self-contained in this notebook — no external library imports
from a shared `src/` package.

In [ ]:
# ---------------------------------------------------------------------------
# Papermill parameters  (this cell is tagged "parameters")
# ---------------------------------------------------------------------------

# Data loading
feature_paths: list[str] = []          # local or gs:// URIs; empty -> synthetic
target_col: str = 'target'

# Splitting
test_size: float = 0.2
random_state: int = 42

# Optuna (per model)
optuna_n_trials: int = 20
optuna_timeout_s: int | None = None
optimize_metric: str = 'f1'            # 'f1', 'precision', 'recall'
threshold_grid: list[float] = [round(x * 0.05, 2) for x in range(1, 20)]

# Outputs
metrics_json_path: str = 'outputs/metrics/metrics.json'
plots_dir: str = 'outputs/plots'
executed_notebook_path: str | None = None

# SHAP / feature importance
shap_sample_size: int = 500
enable_shap: bool = True
enable_feature_importance: bool = True

In [ ]:
# ---------------------------------------------------------------------------
# Imports
# ---------------------------------------------------------------------------
import json
import os
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path

warnings.filterwarnings('ignore')
os.environ['PYTHONWARNINGS'] = 'ignore'

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
from catboost import CatBoostClassifier, Pool
import xgboost as xgb
import lightgbm as lgb
from sklearn.datasets import make_classification
from sklearn.metrics import (
    auc,
    average_precision_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.calibration import calibration_curve
from sklearn.model_selection import StratifiedKFold, train_test_split
from scipy.stats import spearmanr
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

for d in ['outputs/runs', 'outputs/plots', 'outputs/models', 'outputs/metrics']:
    Path(d).mkdir(parents=True, exist_ok=True)

print(f'Run started at {datetime.now(timezone.utc).isoformat()}')

## 1 — Meet the Contenders

Gradient boosting is the backbone of tabular ML. Three frameworks dominate:
CatBoost (Yandex, 2017), XGBoost (Tianqi Chen, 2014), and LightGBM
(Microsoft Research, 2017). They share the core idea — build an ensemble of
decision trees sequentially, each correcting the errors of the previous
ones — but differ significantly in their approaches.

### CatBoost

CatBoost introduced two key innovations:

- **Ordered boosting** — a permutation-driven scheme that computes target
  statistics in a way that avoids prediction shift (a subtle form of
  target leakage). Each training example is scored using a model trained
  only on examples that precede it in a random permutation.
- **Native categorical handling** — CatBoost encodes categoricals using
  ordered target statistics during training. No one-hot encoding needed.
  The encoding is stored in the model file, so at inference time you pass
  raw categorical strings and the model handles everything.

CatBoost also uses **oblivious (symmetric) decision trees**: every node at
the same depth uses the same split condition. This acts as regularization
and enables extremely fast inference (branchless evaluation).

**Deployment advantage:** A saved `.cbm` model remembers feature names,
categorical indices, and all encoding info. Load it and predict — no
separate preprocessing pipeline.

### XGBoost

XGBoost popularized gradient boosting and remains the most widely used
framework. Key aspects:

- **Level-wise tree growth** with explicit regularization: L1 (alpha),
  L2 (lambda), and minimum loss reduction (gamma) for splits.
- **Multiple tree methods:** `exact` (exact greedy), `approx` (quantile
  sketch), and `hist` (histogram-based, now the default). The `hist`
  method is similar to LightGBM's approach.
- **Categorical support** via `enable_categorical=True` (XGBoost >= 1.5).
  Uses an optimal partitioning algorithm for categorical splits. Requires
  pandas `Categorical` dtype columns and `tree_method='hist'`.
- **Multiple feature importance types:** `weight` (number of times a
  feature is used in splits), `gain` (average gain per split), and
  `cover` (average coverage per split). Each tells a different story.

**Deployment note:** XGBoost does not store preprocessing information. You
need to maintain the same one-hot encoding or category mapping at inference.
With `enable_categorical`, you must ensure the same category set.

### LightGBM

LightGBM was designed for speed and handles large datasets efficiently:

- **Leaf-wise tree growth** — instead of growing level-by-level, LightGBM
  always splits the leaf with the largest loss reduction. This often
  converges faster but is more prone to overfitting (controlled via
  `num_leaves`).
- **GOSS (Gradient-based One-Side Sampling)** — keeps instances with large
  gradients and randomly samples those with small gradients, reducing
  computation while maintaining accuracy.
- **EFB (Exclusive Feature Bundling)** — bundles mutually exclusive features
  into single features, reducing dimensionality without information loss.
- **Native categorical support** — specify `categorical_feature` in
  training. LightGBM finds optimal categorical splits using a
  histogram-based algorithm.

**Speed:** LightGBM is generally the fastest of the three, especially on
large datasets. The combination of leaf-wise growth, GOSS, and EFB makes
it highly efficient.

### Comparison at a Glance

| Aspect | CatBoost | XGBoost | LightGBM |
|---|---|---|---|
| Tree growth | Level-wise (symmetric/oblivious) | Level-wise | Leaf-wise |
| Categorical handling | Native (ordered target stats) | `enable_categorical` (hist) | Native (optimal split) |
| One-hot needed? | No | No (with enable_categorical) | No |
| Overfitting control | Ordered boosting + symmetric trees | L1/L2/gamma regularization | `num_leaves` + L1/L2 |
| Training speed | Moderate | Moderate | Fast |
| Default importance type | PredictionValuesChange | Gain | Split count |
| Deployment ease | High (model stores everything) | Lower (need preprocessing) | Medium |
| SHAP support | Built-in + shap library | shap library | shap library |

## 2 — Data Loading

In [ ]:
# ---------------------------------------------------------------------------
# Data loading — synthetic dataset with numeric + categorical features
# ---------------------------------------------------------------------------

def _read_file(path: str) -> pl.DataFrame:
    p = path.strip()
    if p.endswith('.parquet') or p.endswith('.pq'):
        return pl.read_parquet(p)
    return pl.read_csv(p)

if feature_paths:
    dfs = [_read_file(p) for p in feature_paths]
    df = pl.concat(dfs, how='vertical_relaxed')
    print(f'Loaded {len(feature_paths)} file(s): {df.shape[0]:,} rows x {df.shape[1]} cols')
else:
    print('No feature_paths provided — generating synthetic dataset.')
    n_samples = 8_000
    rng = np.random.RandomState(random_state)

    X_num, y = make_classification(
        n_samples=n_samples, n_features=15, n_informative=10,
        n_redundant=3, n_classes=2, weights=[0.7, 0.3],
        flip_y=0.03, random_state=random_state,
    )

    # Categorical features with varying cardinality
    region = rng.choice(['north', 'south', 'east', 'west'], size=n_samples)
    tier = rng.choice(
        ['bronze', 'silver', 'gold', 'platinum'],
        size=n_samples, p=[0.4, 0.3, 0.2, 0.1],
    )
    channel = rng.choice(
        ['web', 'mobile', 'api', 'partner', 'direct'], size=n_samples,
    )

    # Make categoricals somewhat predictive
    boost = ((tier == 'platinum') | (tier == 'gold')) & (rng.random(n_samples) > 0.5)
    y[boost] = 1
    suppress = (region == 'north') & (rng.random(n_samples) > 0.7)
    y[suppress] = 0

    cols = {f'feat_{i:02d}': X_num[:, i] for i in range(X_num.shape[1])}
    cols['region'] = region
    cols['tier'] = tier
    cols['channel'] = channel
    cols[target_col] = y.astype(int)
    df = pl.DataFrame(cols)

# Identify column types
all_feature_cols = [c for c in df.columns if c != target_col]
_NUMERIC_TYPES = {
    pl.Float32, pl.Float64, pl.Int8, pl.Int16, pl.Int32, pl.Int64,
    pl.UInt8, pl.UInt16, pl.UInt32, pl.UInt64,
}
num_cols = [c for c in all_feature_cols if df[c].dtype in _NUMERIC_TYPES]
cat_cols = [c for c in all_feature_cols if c not in num_cols]
feature_names = num_cols + cat_cols

print(f'DataFrame: {df.shape[0]:,} rows x {df.shape[1]} cols')
print(f'Numeric features ({len(num_cols)}): {num_cols[:5]}{"..." if len(num_cols) > 5 else ""}')
print(f'Categorical features ({len(cat_cols)}): {cat_cols}')

## 3 — EDA

In [ ]:
# ---------------------------------------------------------------------------
# Schema, target distribution, histograms, categorical distributions
# ---------------------------------------------------------------------------
print('Schema:')
for name, dtype in zip(df.columns, df.dtypes):
    null_ct = df[name].null_count()
    print(f'  {name:30s}  {str(dtype):15s}  nulls={null_ct}')

print(f'\nTarget distribution ({target_col}):')
print(df[target_col].value_counts().sort(target_col))

# Histograms (first 8 numeric features)
plot_cols = num_cols[:8]
if plot_cols:
    n = len(plot_cols)
    ncols = min(4, n)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3 * nrows))
    axes_flat = np.array(axes).flatten() if n > 1 else [axes]
    for i, col in enumerate(plot_cols):
        axes_flat[i].hist(df[col].drop_nulls().to_numpy(), bins=40, edgecolor='white')
        axes_flat[i].set_title(col, fontsize=10)
    for j in range(i + 1, len(axes_flat)):
        axes_flat[j].set_visible(False)
    fig.tight_layout()
    fig.savefig(f'{plots_dir}/eda_histograms.png', dpi=120)
    plt.show()

# Categorical distributions
if cat_cols:
    n_cat = len(cat_cols)
    fig, axes = plt.subplots(1, n_cat, figsize=(5 * n_cat, 4))
    if n_cat == 1:
        axes = [axes]
    for ax, col in zip(axes, cat_cols):
        vc = df[col].value_counts().sort(col)
        ax.bar(vc[col].to_list(), vc['count'].to_list(), edgecolor='white')
        ax.set_title(col)
        ax.tick_params(axis='x', rotation=45)
    fig.tight_layout()
    fig.savefig(f'{plots_dir}/eda_categoricals.png', dpi=120)
    plt.show()

## 4 — Train / Test Split & Data Preparation

We split once and prepare framework-specific data formats. Each framework
handles categorical features differently:

- **CatBoost:** Pass a pandas DataFrame with string columns + specify
  `cat_features`. CatBoost handles encoding internally.
- **XGBoost:** Convert categorical columns to pandas `Categorical` dtype +
  set `enable_categorical=True`.
- **LightGBM:** Convert categorical columns to pandas `Categorical` dtype +
  specify `categorical_feature` in `.fit()`.

In [ ]:
# ---------------------------------------------------------------------------
# Train / test split and framework-specific data preparation
# ---------------------------------------------------------------------------
X_all_pd = df.select(feature_names).to_pandas()
y_all = df[target_col].to_numpy().astype(int)

X_train_pd, X_test_pd, y_train, y_test = train_test_split(
    X_all_pd, y_all, test_size=test_size, random_state=random_state, stratify=y_all,
)
X_train_pd = X_train_pd.reset_index(drop=True)
X_test_pd = X_test_pd.reset_index(drop=True)

# Class imbalance
n_neg = int((y_train == 0).sum())
n_pos = int((y_train == 1).sum())
scale_pos_weight = n_neg / n_pos

print(f'Train: {X_train_pd.shape[0]:,}  |  Test: {X_test_pd.shape[0]:,}')
print(f'Train target rate: {y_train.mean():.3f}  |  Test target rate: {y_test.mean():.3f}')
print(f'scale_pos_weight: {scale_pos_weight:.4f}')

# -- CatBoost: string categoricals, specify cat_features ------------------
# CatBoost can take the DataFrame as-is with cat_features
print('\n[CatBoost] Using raw string categoricals with cat_features parameter')

# -- XGBoost: pandas Categorical dtype + enable_categorical ----------------
X_train_xgb = X_train_pd.copy()
X_test_xgb = X_test_pd.copy()
for c in cat_cols:
    cat_type = pd.CategoricalDtype(categories=sorted(X_train_pd[c].unique()))
    X_train_xgb[c] = X_train_xgb[c].astype(cat_type)
    X_test_xgb[c] = X_test_xgb[c].astype(cat_type)
print(f'[XGBoost] Converted {len(cat_cols)} columns to pandas Categorical dtype')

# -- LightGBM: pandas Categorical dtype + categorical_feature -------------
X_train_lgb = X_train_pd.copy()
X_test_lgb = X_test_pd.copy()
for c in cat_cols:
    cat_type = pd.CategoricalDtype(categories=sorted(X_train_pd[c].unique()))
    X_train_lgb[c] = X_train_lgb[c].astype(cat_type)
    X_test_lgb[c] = X_test_lgb[c].astype(cat_type)
print(f'[LightGBM] Converted {len(cat_cols)} columns to pandas Categorical dtype')

## 5 — CatBoost Deep Dive

CatBoost's killer features for this comparison:

1. **Zero preprocessing** — pass raw string categoricals via `cat_features`
2. **Ordered boosting** — reduces overfitting without explicit regularization tuning
3. **`use_best_model=True`** — automatic early stopping using an eval set
4. **Model serialization** — `.save_model()` stores feature names, categorical
   indices, and all target-statistic encoding. At inference, just
   `model.load_model()` and predict on raw data.

CatBoost's hyperparameter search space focuses on tree structure and
regularization, but notably does **not** require tuning categorical encoding
parameters — it handles that automatically.

In [ ]:
# ---------------------------------------------------------------------------
# CatBoost — Optuna hyperparameter tuning + final model
# ---------------------------------------------------------------------------
_METRIC_FN = {
    'f1': f1_score,
    'precision': precision_score,
    'recall': recall_score,
}


def cb_objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 200, 1500),
        'depth': trial.suggest_int('depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-2, 10.0, log=True),
        'border_count': trial.suggest_int('border_count', 32, 255),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'random_strength': trial.suggest_float('random_strength', 1e-3, 10.0, log=True),
        'scale_pos_weight': scale_pos_weight,
    }
    metric_fn = _METRIC_FN[optimize_metric]
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=random_state)
    fold_scores = []
    for tr_idx, vl_idx in skf.split(X_train_pd, y_train):
        model = CatBoostClassifier(
            **params, cat_features=cat_cols, eval_metric='Logloss',
            random_seed=random_state, verbose=0, early_stopping_rounds=50,
        )
        model.fit(
            X_train_pd.iloc[tr_idx], y_train[tr_idx],
            eval_set=(X_train_pd.iloc[vl_idx], y_train[vl_idx]), verbose=0,
        )
        proba = model.predict_proba(X_train_pd.iloc[vl_idx])[:, 1]
        best = max(
            metric_fn(y_train[vl_idx], (proba >= t).astype(int), zero_division=0)
            for t in threshold_grid
        )
        fold_scores.append(best)
    return float(np.mean(fold_scores))


print('Tuning CatBoost...')
cb_study = optuna.create_study(direction='maximize')
cb_study.optimize(cb_objective, n_trials=optuna_n_trials, timeout=optuna_timeout_s)

cb_best_params = cb_study.best_params
print(f'Best CV {optimize_metric}: {cb_study.best_value:.4f}')
print(f'Best params: {json.dumps(cb_best_params, indent=2)}')

# Train final CatBoost model
print('\nTraining final CatBoost model...')
cb_model = CatBoostClassifier(
    **cb_best_params, cat_features=cat_cols, scale_pos_weight=scale_pos_weight,
    eval_metric='Logloss', random_seed=random_state, verbose=100,
    use_best_model=True, early_stopping_rounds=100,
)
cb_t0 = time.time()
cb_model.fit(X_train_pd, y_train, eval_set=(X_test_pd, y_test), verbose=100)
cb_train_time = time.time() - cb_t0

cb_model.save_model('outputs/models/catboost_model.cbm')
print(f'\nCatBoost training time: {cb_train_time:.2f}s')
print(f'Best iteration: {cb_model.best_iteration_}')
print(f'Model saved -> outputs/models/catboost_model.cbm')

In [ ]:
# ---------------------------------------------------------------------------
# CatBoost — bells & whistles
# ---------------------------------------------------------------------------

# -- Learning curve (overfitting detection) --------------------------------
evals = cb_model.get_evals_result()
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(evals['learn']['Logloss'], label='Train', alpha=0.8)
ax.plot(evals['validation']['Logloss'], label='Validation', alpha=0.8)
if cb_model.best_iteration_ is not None:
    ax.axvline(cb_model.best_iteration_, color='red', linestyle='--', alpha=0.7,
               label=f'Best iteration ({cb_model.best_iteration_})')
ax.set_xlabel('Iteration')
ax.set_ylabel('Logloss')
ax.set_title('CatBoost — Learning Curve (Overfitting Detection)')
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(f'{plots_dir}/cb_learning_curve.png', dpi=120)
plt.show()

# -- Model remembers everything for deployment -----------------------------
loaded = CatBoostClassifier()
loaded.load_model('outputs/models/catboost_model.cbm')
print('Loaded model feature names:', loaded.feature_names_[:5], '...')
print('Loaded model cat feature indices:', loaded.get_cat_feature_indices())
print('-> CatBoost model stores feature names + categorical encoding.')
print('   At inference: just pass raw data (strings OK). No preprocessing needed.')

# -- Feature importance (PredictionValuesChange) ---------------------------
if enable_feature_importance:
    cb_importance = cb_model.get_feature_importance()
    cb_imp_df = (
        pl.DataFrame({'feature': feature_names, 'importance': cb_importance})
        .sort('importance', descending=True)
    )
    top_n = min(20, len(cb_imp_df))
    top = cb_imp_df.head(top_n)
    fig, ax = plt.subplots(figsize=(8, max(4, top_n * 0.35)))
    ax.barh(top['feature'].to_list()[::-1], top['importance'].to_list()[::-1], color='#2196F3')
    ax.set_xlabel('Importance (PredictionValuesChange)')
    ax.set_title('CatBoost — Feature Importance')
    fig.tight_layout()
    fig.savefig(f'{plots_dir}/cb_feature_importance.png', dpi=120)
    plt.show()

## 6 — XGBoost Deep Dive

XGBoost is the most battle-tested framework. Key points for this comparison:

1. **`enable_categorical=True`** — newer feature (XGBoost >= 1.5) that handles
   categoricals via optimal partitioning. Requires `tree_method='hist'` and
   pandas `Categorical` dtype columns.
2. **Explicit regularization** — `reg_alpha` (L1), `reg_lambda` (L2), and
   `gamma` (minimum loss reduction for splits) give fine-grained control.
3. **Multiple importance types** — `weight` (split count), `gain` (average gain),
   `cover` (average coverage). XGBoost is the only framework that makes all
   three easily accessible.
4. **DMatrix** — XGBoost's internal data structure for efficient computation.
   The sklearn API wraps this, but you can use it directly for more control.

**Preprocessing note:** With `enable_categorical`, you must convert categorical
columns to pandas `Categorical` dtype before training. The category set
from training must be maintained at inference time — this is your
responsibility, unlike CatBoost where the model handles it.

In [ ]:
# ---------------------------------------------------------------------------
# XGBoost — Optuna hyperparameter tuning + final model
# ---------------------------------------------------------------------------


def xgb_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 1500),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'gamma': trial.suggest_float('gamma', 1e-3, 5.0, log=True),
        'scale_pos_weight': scale_pos_weight,
    }
    metric_fn = _METRIC_FN[optimize_metric]
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=random_state)
    fold_scores = []
    for tr_idx, vl_idx in skf.split(X_train_xgb, y_train):
        model = xgb.XGBClassifier(
            **params, enable_categorical=True, tree_method='hist',
            eval_metric='logloss', random_state=random_state,
            early_stopping_rounds=50, verbosity=0,
        )
        model.fit(
            X_train_xgb.iloc[tr_idx], y_train[tr_idx],
            eval_set=[(X_train_xgb.iloc[vl_idx], y_train[vl_idx])], verbose=False,
        )
        proba = model.predict_proba(X_train_xgb.iloc[vl_idx])[:, 1]
        best = max(
            metric_fn(y_train[vl_idx], (proba >= t).astype(int), zero_division=0)
            for t in threshold_grid
        )
        fold_scores.append(best)
    return float(np.mean(fold_scores))


print('Tuning XGBoost...')
xgb_study = optuna.create_study(direction='maximize')
xgb_study.optimize(xgb_objective, n_trials=optuna_n_trials, timeout=optuna_timeout_s)

xgb_best_params = xgb_study.best_params
print(f'Best CV {optimize_metric}: {xgb_study.best_value:.4f}')
print(f'Best params: {json.dumps(xgb_best_params, indent=2)}')

# Train final XGBoost model
print('\nTraining final XGBoost model...')
xgb_model = xgb.XGBClassifier(
    **xgb_best_params, enable_categorical=True, tree_method='hist',
    scale_pos_weight=scale_pos_weight, eval_metric='logloss',
    random_state=random_state, early_stopping_rounds=100, verbosity=0,
)
xgb_t0 = time.time()
xgb_model.fit(
    X_train_xgb, y_train,
    eval_set=[(X_train_xgb, y_train), (X_test_xgb, y_test)], verbose=False,
)
xgb_train_time = time.time() - xgb_t0

xgb_model.save_model('outputs/models/xgboost_model.ubj')
print(f'\nXGBoost training time: {xgb_train_time:.2f}s')
print(f'Best iteration: {xgb_model.best_iteration}')
print(f'Model saved -> outputs/models/xgboost_model.ubj')

In [ ]:
# ---------------------------------------------------------------------------
# XGBoost — bells & whistles
# ---------------------------------------------------------------------------

# -- Learning curve --------------------------------------------------------
xgb_evals = xgb_model.evals_result()
xgb_eval_keys = list(xgb_evals.keys())
xgb_metric_key = list(xgb_evals[xgb_eval_keys[0]].keys())[0]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(xgb_evals[xgb_eval_keys[0]][xgb_metric_key], label='Train', alpha=0.8)
ax.plot(xgb_evals[xgb_eval_keys[1]][xgb_metric_key], label='Validation', alpha=0.8)
if xgb_model.best_iteration is not None:
    ax.axvline(xgb_model.best_iteration, color='red', linestyle='--', alpha=0.7,
               label=f'Best iteration ({xgb_model.best_iteration})')
ax.set_xlabel('Iteration')
ax.set_ylabel(xgb_metric_key)
ax.set_title('XGBoost — Learning Curve')
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(f'{plots_dir}/xgb_learning_curve.png', dpi=120)
plt.show()

# -- Multiple feature importance types ------------------------------------
if enable_feature_importance:
    imp_types = ['weight', 'gain', 'cover']
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    for ax, imp_type in zip(axes, imp_types):
        booster = xgb_model.get_booster()
        scores = booster.get_score(importance_type=imp_type)
        if scores:
            sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:15]
            names = [s[0] for s in sorted_scores][::-1]
            vals = [s[1] for s in sorted_scores][::-1]
            ax.barh(names, vals, color='#4CAF50')
        ax.set_xlabel(imp_type.capitalize())
        ax.set_title(f'XGBoost — {imp_type.capitalize()} Importance')
    fig.tight_layout()
    fig.savefig(f'{plots_dir}/xgb_feature_importance.png', dpi=120)
    plt.show()
    print('XGBoost offers 3 importance types:')
    print('  weight — how many times a feature is used in splits')
    print('  gain   — average improvement per split (closest to CatBoost default)')
    print('  cover  — average number of samples affected per split')

## 7 — LightGBM Deep Dive

LightGBM's standout qualities for this comparison:

1. **Speed** — leaf-wise growth + GOSS + EFB make it typically the fastest
   to train, often by a significant margin on larger datasets.
2. **`num_leaves`** — the primary complexity parameter. Unlike `max_depth` in
   CatBoost/XGBoost, LightGBM controls complexity via the number of leaves.
   A tree with `num_leaves=31` can be deeper than `max_depth=5` because
   leaf-wise growth produces asymmetric trees.
3. **Native categoricals** — specify `categorical_feature` in `.fit()` or use
   pandas `Categorical` dtype. LightGBM finds optimal splits on categories.
4. **`lgb.cv()`** — built-in cross-validation with early stopping, useful for
   quick baseline validation.

**Overfitting note:** Leaf-wise growth is more prone to overfitting than
level-wise growth, especially on small datasets. The `num_leaves` parameter
is critical — set it lower than `2^max_depth` for regularization.

In [ ]:
# ---------------------------------------------------------------------------
# LightGBM — Optuna hyperparameter tuning + final model
# ---------------------------------------------------------------------------


def lgb_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 1500),
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 300),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'scale_pos_weight': scale_pos_weight,
    }
    metric_fn = _METRIC_FN[optimize_metric]
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=random_state)
    fold_scores = []
    for tr_idx, vl_idx in skf.split(X_train_lgb, y_train):
        model = lgb.LGBMClassifier(**params, random_state=random_state, verbose=-1)
        model.fit(
            X_train_lgb.iloc[tr_idx], y_train[tr_idx],
            eval_set=[(X_train_lgb.iloc[vl_idx], y_train[vl_idx])],
            callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)],
        )
        proba = model.predict_proba(X_train_lgb.iloc[vl_idx])[:, 1]
        best = max(
            metric_fn(y_train[vl_idx], (proba >= t).astype(int), zero_division=0)
            for t in threshold_grid
        )
        fold_scores.append(best)
    return float(np.mean(fold_scores))


print('Tuning LightGBM...')
lgb_study = optuna.create_study(direction='maximize')
lgb_study.optimize(lgb_objective, n_trials=optuna_n_trials, timeout=optuna_timeout_s)

lgb_best_params = lgb_study.best_params
print(f'Best CV {optimize_metric}: {lgb_study.best_value:.4f}')
print(f'Best params: {json.dumps(lgb_best_params, indent=2)}')

# Train final LightGBM model
print('\nTraining final LightGBM model...')
lgb_model = lgb.LGBMClassifier(
    **lgb_best_params, scale_pos_weight=scale_pos_weight,
    random_state=random_state, verbose=-1,
)
lgb_t0 = time.time()
lgb_model.fit(
    X_train_lgb, y_train,
    eval_set=[(X_train_lgb, y_train), (X_test_lgb, y_test)],
    callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)],
)
lgb_train_time = time.time() - lgb_t0

lgb_model.booster_.save_model('outputs/models/lightgbm_model.lgb')
print(f'\nLightGBM training time: {lgb_train_time:.2f}s')
print(f'Best iteration: {lgb_model.best_iteration_}')
print(f'Model saved -> outputs/models/lightgbm_model.lgb')

In [ ]:
# ---------------------------------------------------------------------------
# LightGBM — bells & whistles
# ---------------------------------------------------------------------------

# -- Learning curve --------------------------------------------------------
lgb_evals = lgb_model.evals_result_
lgb_eval_keys = list(lgb_evals.keys())
lgb_metric_key = list(lgb_evals[lgb_eval_keys[0]].keys())[0]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(lgb_evals[lgb_eval_keys[0]][lgb_metric_key], label='Train', alpha=0.8)
ax.plot(lgb_evals[lgb_eval_keys[1]][lgb_metric_key], label='Validation', alpha=0.8)
if lgb_model.best_iteration_ is not None:
    ax.axvline(lgb_model.best_iteration_, color='red', linestyle='--', alpha=0.7,
               label=f'Best iteration ({lgb_model.best_iteration_})')
ax.set_xlabel('Iteration')
ax.set_ylabel(lgb_metric_key)
ax.set_title('LightGBM — Learning Curve')
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(f'{plots_dir}/lgb_learning_curve.png', dpi=120)
plt.show()

# -- Feature importance: split vs gain ------------------------------------
if enable_feature_importance:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    for ax, imp_type in zip(axes, ['split', 'gain']):
        importances = lgb_model.booster_.feature_importance(importance_type=imp_type)
        imp_df = (
            pl.DataFrame({'feature': feature_names, 'importance': importances.astype(float)})
            .sort('importance', descending=True)
        )
        top_n = min(15, len(imp_df))
        top = imp_df.head(top_n)
        ax.barh(top['feature'].to_list()[::-1], top['importance'].to_list()[::-1], color='#FF9800')
        ax.set_xlabel(imp_type.capitalize())
        ax.set_title(f'LightGBM — {imp_type.capitalize()} Importance')
    fig.tight_layout()
    fig.savefig(f'{plots_dir}/lgb_feature_importance.png', dpi=120)
    plt.show()
    print('LightGBM offers 2 importance types:')
    print('  split — how often a feature is used (can be misleading)')
    print('  gain  — total improvement from splits on this feature')

## 8 — Head-to-Head: Performance Comparison

Now we compare all three models on the same test set using identical
threshold analysis. The evaluation is completely fair — same data split,
same Optuna budget, same metrics.

In [ ]:
# ---------------------------------------------------------------------------
# Compute predictions and threshold metrics for all three models
# ---------------------------------------------------------------------------
models_eval = {
    'CatBoost': (cb_model, X_test_pd),
    'XGBoost': (xgb_model, X_test_xgb),
    'LightGBM': (lgb_model, X_test_lgb),
}
colors = {'CatBoost': '#2196F3', 'XGBoost': '#4CAF50', 'LightGBM': '#FF9800'}

results = {}
for name, (model, X_test_fmt) in models_eval.items():
    y_proba = model.predict_proba(X_test_fmt)[:, 1]
    rows = []
    for t in threshold_grid:
        y_pred_t = (y_proba >= t).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_test, y_pred_t, labels=[0, 1]).ravel()
        rows.append({
            'threshold': round(t, 4),
            'precision': round(precision_score(y_test, y_pred_t, zero_division=0), 4),
            'recall': round(recall_score(y_test, y_pred_t, zero_division=0), 4),
            'f1': round(f1_score(y_test, y_pred_t, zero_division=0), 4),
            'tp': int(tp), 'fp': int(fp), 'tn': int(tn), 'fn': int(fn),
        })
    thr_df = pl.DataFrame(rows)
    best_row = thr_df.sort(optimize_metric, descending=True).row(0, named=True)
    roc_auc_val = roc_auc_score(y_test, y_proba)
    pr_auc_val = average_precision_score(y_test, y_proba)
    results[name] = {
        'y_proba': y_proba,
        'threshold_df': thr_df,
        'threshold_rows': rows,
        'best_row': best_row,
        'best_threshold': best_row['threshold'],
        'roc_auc': roc_auc_val,
        'pr_auc': pr_auc_val,
    }
    print(f'{name:12s}  best_thr={best_row["threshold"]:.2f}  '
          f'F1={best_row["f1"]:.4f}  P={best_row["precision"]:.4f}  '
          f'R={best_row["recall"]:.4f}  ROC-AUC={roc_auc_val:.4f}  '
          f'PR-AUC={pr_auc_val:.4f}')

In [ ]:
# ---------------------------------------------------------------------------
# Training time comparison + confusion matrices
# ---------------------------------------------------------------------------
train_times = {'CatBoost': cb_train_time, 'XGBoost': xgb_train_time, 'LightGBM': lgb_train_time}

fig, axes = plt.subplots(1, 4, figsize=(22, 5))

# Training time bar chart
ax = axes[0]
bars = ax.bar(train_times.keys(), train_times.values(),
              color=[colors[k] for k in train_times.keys()], edgecolor='white')
for bar, t in zip(bars, train_times.values()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1,
            f'{t:.1f}s', ha='center', va='bottom', fontsize=10)
ax.set_ylabel('Seconds')
ax.set_title('Training Time')
ax.grid(True, alpha=0.3, axis='y')

# Confusion matrices
for i, (name, res) in enumerate(results.items()):
    ax = axes[i + 1]
    y_pred = (res['y_proba'] >= res['best_threshold']).astype(int)
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax, cmap='Blues')
    ax.set_title(f'{name}\n(thr={res["best_threshold"]:.2f})')

fig.tight_layout()
fig.savefig(f'{plots_dir}/training_time_confusion.png', dpi=120)
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# ROC and PR curves overlaid
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ROC curves
ax = axes[0]
for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res['y_proba'])
    ax.plot(fpr, tpr, linewidth=2, label=f'{name} (AUC={res["roc_auc"]:.4f})',
            color=colors[name])
ax.plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)

# PR curves
ax = axes[1]
prevalence = y_test.mean()
for name, res in results.items():
    pr_prec, pr_rec, _ = precision_recall_curve(y_test, res['y_proba'])
    ax.plot(pr_rec, pr_prec, linewidth=2, label=f'{name} (AP={res["pr_auc"]:.4f})',
            color=colors[name])
ax.axhline(prevalence, color='k', linestyle='--', alpha=0.4,
           label=f'Baseline ({prevalence:.3f})')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curves')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.05)

fig.tight_layout()
fig.savefig(f'{plots_dir}/roc_pr_curves.png', dpi=120)
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Precision / Recall / F1 vs Threshold (one subplot per model)
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
for ax, (name, res) in zip(axes, results.items()):
    thr_df = res['threshold_df']
    thresholds = thr_df['threshold'].to_list()
    ax.plot(thresholds, thr_df['precision'].to_list(), label='Precision', marker='.', color='#4CAF50')
    ax.plot(thresholds, thr_df['recall'].to_list(), label='Recall', marker='.', color='#FF9800')
    ax.plot(thresholds, thr_df['f1'].to_list(), label='F1', marker='.', linewidth=2, color='#2196F3')
    ax.axvline(res['best_threshold'], color='red', linestyle='--', alpha=0.7,
               label=f'Best thr={res["best_threshold"]}')
    ax.set_xlabel('Threshold')
    ax.set_ylabel('Score')
    ax.set_title(f'{name} — Metrics vs Threshold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(-0.05, 1.05)
fig.tight_layout()
fig.savefig(f'{plots_dir}/threshold_per_model.png', dpi=120)
plt.show()

# ---------------------------------------------------------------------------
# Calibration curves overlaid
# ---------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(7, 6))
for name, res in results.items():
    prob_true, prob_pred = calibration_curve(y_test, res['y_proba'], n_bins=10, strategy='uniform')
    ax.plot(prob_pred, prob_true, 's-', linewidth=2, label=name, color=colors[name])
ax.plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Perfectly calibrated')
ax.set_xlabel('Mean Predicted Probability')
ax.set_ylabel('Fraction of Positives')
ax.set_title('Calibration Curves')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(f'{plots_dir}/calibration_curves.png', dpi=120)
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Predicted probability distributions
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
mask_neg = y_test == 0
mask_pos = y_test == 1
for ax, (name, res) in zip(axes, results.items()):
    ax.hist(res['y_proba'][mask_neg], bins=50, alpha=0.6, label='Class 0',
            color='#2196F3', edgecolor='white')
    ax.hist(res['y_proba'][mask_pos], bins=50, alpha=0.6, label='Class 1',
            color='#FF5722', edgecolor='white')
    ax.axvline(res['best_threshold'], color='red', linestyle='--', linewidth=2,
               label=f'thr={res["best_threshold"]}')
    ax.set_xlabel('Predicted Probability')
    ax.set_ylabel('Count')
    ax.set_title(f'{name}')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
fig.suptitle('Predicted Probability Distribution by Class', fontsize=14, y=1.02)
fig.tight_layout()
fig.savefig(f'{plots_dir}/probability_distributions.png', dpi=120, bbox_inches='tight')
plt.show()

# ---------------------------------------------------------------------------
# Summary comparison table
# ---------------------------------------------------------------------------
print('\n' + '=' * 90)
print(f'{"Metric":<25} {"CatBoost":>15} {"XGBoost":>15} {"LightGBM":>15}   {"Winner":>12}')
print('=' * 90)

metrics_compare = [
    ('Best F1', 'f1'),
    ('Precision @ best thr', 'precision'),
    ('Recall @ best thr', 'recall'),
]
for label, key in metrics_compare:
    vals = {name: res['best_row'][key] for name, res in results.items()}
    winner = max(vals, key=vals.get)
    print(f'{label:<25} {vals["CatBoost"]:>15.4f} {vals["XGBoost"]:>15.4f} '
          f'{vals["LightGBM"]:>15.4f}   {winner:>12}')

for label, key in [('ROC-AUC', 'roc_auc'), ('PR-AUC', 'pr_auc')]:
    vals = {name: res[key] for name, res in results.items()}
    winner = max(vals, key=vals.get)
    print(f'{label:<25} {vals["CatBoost"]:>15.4f} {vals["XGBoost"]:>15.4f} '
          f'{vals["LightGBM"]:>15.4f}   {winner:>12}')

vals = train_times
winner = min(vals, key=vals.get)
print(f'{"Training Time (s)":<25} {vals["CatBoost"]:>15.1f} {vals["XGBoost"]:>15.1f} '
      f'{vals["LightGBM"]:>15.1f}   {winner:>12}')

best_thrs = {name: res['best_threshold'] for name, res in results.items()}
print(f'{"Best Threshold":<25} {best_thrs["CatBoost"]:>15.2f} {best_thrs["XGBoost"]:>15.2f} '
      f'{best_thrs["LightGBM"]:>15.2f}')
print('=' * 90)

## 9 — Feature Importance & SHAP Comparison

We compare what each model says about feature importance and SHAP values.
All three use TreeSHAP (via the `shap` library) for consistent comparison.
Do they agree on which features matter most? Disagreements reveal
differences in how each framework models the data.

In [ ]:
# ---------------------------------------------------------------------------
# Side-by-side feature importance (each model's default importance type)
# ---------------------------------------------------------------------------
if enable_feature_importance:
    fig, axes = plt.subplots(1, 3, figsize=(20, 7))

    # CatBoost — PredictionValuesChange
    cb_imp = cb_model.get_feature_importance()
    cb_imp_order = np.argsort(cb_imp)[-15:]
    ax = axes[0]
    ax.barh([feature_names[i] for i in cb_imp_order], cb_imp[cb_imp_order], color=colors['CatBoost'])
    ax.set_title('CatBoost\n(PredictionValuesChange)')
    ax.set_xlabel('Importance')

    # XGBoost — Gain (default for sklearn API)
    xgb_imp = xgb_model.feature_importances_
    xgb_imp_order = np.argsort(xgb_imp)[-15:]
    ax = axes[1]
    ax.barh([feature_names[i] for i in xgb_imp_order], xgb_imp[xgb_imp_order], color=colors['XGBoost'])
    ax.set_title('XGBoost\n(Gain)')
    ax.set_xlabel('Importance')

    # LightGBM — Split (default for sklearn API)
    lgb_imp = lgb_model.feature_importances_
    lgb_imp_order = np.argsort(lgb_imp)[-15:]
    ax = axes[2]
    ax.barh([feature_names[i] for i in lgb_imp_order], lgb_imp[lgb_imp_order], color=colors['LightGBM'])
    ax.set_title('LightGBM\n(Split Count)')
    ax.set_xlabel('Importance')

    fig.suptitle('Feature Importance — Default Type per Framework', fontsize=14, y=1.02)
    fig.tight_layout()
    fig.savefig(f'{plots_dir}/feature_importance_comparison.png', dpi=120, bbox_inches='tight')
    plt.show()

    # Rank correlation
    cb_ranks = np.argsort(np.argsort(-cb_imp))
    xgb_ranks = np.argsort(np.argsort(-xgb_imp))
    lgb_ranks = np.argsort(np.argsort(-lgb_imp))
    r_cb_xgb, _ = spearmanr(cb_ranks, xgb_ranks)
    r_cb_lgb, _ = spearmanr(cb_ranks, lgb_ranks)
    r_xgb_lgb, _ = spearmanr(xgb_ranks, lgb_ranks)
    print(f'Feature importance rank correlation (Spearman):')
    print(f'  CatBoost vs XGBoost:  {r_cb_xgb:.3f}')
    print(f'  CatBoost vs LightGBM: {r_cb_lgb:.3f}')
    print(f'  XGBoost vs LightGBM:  {r_xgb_lgb:.3f}')
else:
    print('Feature importance disabled.')

In [ ]:
# ---------------------------------------------------------------------------
# SHAP comparison — TreeSHAP for all three models
# ---------------------------------------------------------------------------
if enable_shap:
    import shap

    rng_shap = np.random.RandomState(random_state)
    n_shap = min(shap_sample_size, X_test_pd.shape[0])
    shap_idx = rng_shap.choice(X_test_pd.shape[0], size=n_shap, replace=False)

    # -- CatBoost SHAP -----------------------------------------------------
    print('Computing CatBoost SHAP values...')
    cb_explainer = shap.TreeExplainer(cb_model)
    cb_pool = Pool(X_test_pd.iloc[shap_idx], cat_features=cat_cols)
    cb_shap_values = cb_explainer.shap_values(cb_pool)

    # -- XGBoost SHAP ------------------------------------------------------
    print('Computing XGBoost SHAP values...')
    xgb_explainer = shap.TreeExplainer(xgb_model)
    xgb_shap_values = xgb_explainer.shap_values(X_test_xgb.iloc[shap_idx])

    # -- LightGBM SHAP ----------------------------------------------------
    print('Computing LightGBM SHAP values...')
    lgb_explainer = shap.TreeExplainer(lgb_model)
    lgb_shap_raw = lgb_explainer.shap_values(X_test_lgb.iloc[shap_idx])
    # LightGBM binary classification may return [class0, class1]
    lgb_shap_values = lgb_shap_raw[1] if isinstance(lgb_shap_raw, list) else lgb_shap_raw

    # -- Summary bar plots (mean |SHAP|) -----------------------------------
    fig, axes = plt.subplots(1, 3, figsize=(20, 7))
    for ax, (name, sv) in zip(axes, [
        ('CatBoost', cb_shap_values),
        ('XGBoost', xgb_shap_values),
        ('LightGBM', lgb_shap_values),
    ]):
        mean_abs_shap = np.abs(sv).mean(axis=0)
        top_idx = np.argsort(mean_abs_shap)[-15:]
        ax.barh(
            [feature_names[i] for i in top_idx],
            mean_abs_shap[top_idx],
            color=colors[name],
        )
        ax.set_xlabel('mean |SHAP value|')
        ax.set_title(f'{name} — SHAP Importance')
    fig.suptitle('SHAP Feature Importance (mean |SHAP value|)', fontsize=14, y=1.02)
    fig.tight_layout()
    fig.savefig(f'{plots_dir}/shap_comparison.png', dpi=120, bbox_inches='tight')
    plt.show()

    # -- SHAP rank correlation ---------------------------------------------
    cb_shap_imp = np.abs(cb_shap_values).mean(axis=0)
    xgb_shap_imp = np.abs(xgb_shap_values).mean(axis=0)
    lgb_shap_imp = np.abs(lgb_shap_values).mean(axis=0)

    r1, _ = spearmanr(cb_shap_imp, xgb_shap_imp)
    r2, _ = spearmanr(cb_shap_imp, lgb_shap_imp)
    r3, _ = spearmanr(xgb_shap_imp, lgb_shap_imp)
    print(f'\nSHAP importance rank correlation (Spearman):')
    print(f'  CatBoost vs XGBoost:  {r1:.3f}')
    print(f'  CatBoost vs LightGBM: {r2:.3f}')
    print(f'  XGBoost vs LightGBM:  {r3:.3f}')

    # -- Top-5 features per model ------------------------------------------
    print(f'\nTop-5 features by SHAP:')
    for name, imp in [('CatBoost', cb_shap_imp), ('XGBoost', xgb_shap_imp), ('LightGBM', lgb_shap_imp)]:
        top5 = np.argsort(imp)[-5:][::-1]
        feats = ', '.join(feature_names[i] for i in top5)
        print(f'  {name:12s}: {feats}')
else:
    print('SHAP disabled.')

## 10 — Practical Considerations & When to Use Which

### Categorical Features: The Great Differentiator

This is where CatBoost truly shines. Consider the full ML lifecycle:

1. **Training:** All three now handle categoricals natively — no one-hot
   encoding needed. CatBoost uses ordered target statistics, XGBoost uses
   optimal partitioning (with `enable_categorical`), and LightGBM uses
   histogram-based optimal splits.

2. **Deployment:** This is where things diverge significantly:
   - **CatBoost:** The `.cbm` model file stores everything — feature names,
     categorical indices, target-statistic encodings. Load it, pass raw
     strings, get predictions. Zero preprocessing code at inference.
   - **XGBoost:** With `enable_categorical`, you need to ensure that
     categorical columns at inference have the same `Categorical` dtype
     with the same categories (in the same order). This means maintaining
     a preprocessing pipeline alongside the model.
   - **LightGBM:** Similar to XGBoost — categorical handling is part of the
     training configuration, not stored with the model in a way that makes
     inference completely preprocessing-free.

### Overfitting Detection

- **CatBoost:** `use_best_model=True` + `eval_set` = automatic. The model
  reverts to the iteration with the best eval metric. Ordered boosting also
  inherently reduces overfitting.
- **XGBoost:** `early_stopping_rounds` in `.fit()` works well. Regularization
  params (`reg_alpha`, `reg_lambda`, `gamma`) give fine-grained control.
- **LightGBM:** `lgb.early_stopping()` callback. But leaf-wise growth is
  more prone to overfitting — requires careful `num_leaves` tuning.

### Speed

LightGBM is typically fastest, especially as dataset size grows. The
GOSS + EFB optimizations shine on large datasets with many features.
CatBoost is typically slowest due to ordered boosting. XGBoost with
`tree_method='hist'` is between the two.

### When to Use Which

| Scenario | Recommendation |
|---|---|
| Many categorical features | CatBoost (best native handling + deployment) |
| Large datasets (>1M rows) | LightGBM (fastest training) |
| Need maximum control over regularization | XGBoost (most explicit params) |
| Quick deployment with minimal code | CatBoost (model stores everything) |
| GPU training | XGBoost or LightGBM (more mature GPU support) |
| Kaggle competition | Try all three + ensemble |
| Production system with string features | CatBoost (no preprocessing at inference) |

### The Verdict

For most tabular ML workflows — especially those with categorical features
and a deployment phase — **CatBoost is the most practical choice**. Its
native categorical handling eliminates an entire class of preprocessing
bugs, and the saved model is truly self-contained. The performance
difference between the three frameworks is usually small (within 1-2% AUC),
so deployment simplicity often matters more than marginal accuracy gains.

If training speed is critical (large datasets, rapid iteration), LightGBM
is hard to beat. If you need maximum control over regularization or
have a purely numeric dataset, XGBoost's battle-tested approach works well.

In practice, the best approach is often to try all three with the same
Optuna budget (as this notebook does) and pick the best for your use case.

## 11 — Metrics JSON Output

In [ ]:
# ---------------------------------------------------------------------------
# Structured metrics JSON — all three models
# ---------------------------------------------------------------------------
metrics_output = {
    'run_metadata': {
        'timestamp': datetime.now(timezone.utc).isoformat(),
        'target_col': target_col,
        'test_size': test_size,
        'random_state': random_state,
        'optuna_n_trials': optuna_n_trials,
        'optimize_metric': optimize_metric,
        'n_train': int(X_train_pd.shape[0]),
        'n_test': int(X_test_pd.shape[0]),
        'n_features': len(feature_names),
        'feature_names': feature_names,
        'cat_features': cat_cols,
        'num_features': num_cols,
        'scale_pos_weight': round(scale_pos_weight, 4),
    },
    'models': {},
}

model_meta = [
    ('catboost', 'CatBoost', cb_study, cb_train_time, cb_best_params),
    ('xgboost', 'XGBoost', xgb_study, xgb_train_time, xgb_best_params),
    ('lightgbm', 'LightGBM', lgb_study, lgb_train_time, lgb_best_params),
]

for key, display_name, study, train_t, best_params in model_meta:
    res = results[display_name]
    metrics_output['models'][key] = {
        'best_optuna_cv_score': round(study.best_value, 4),
        'best_params': best_params,
        'training_time_s': round(train_t, 2),
        'best_threshold': res['best_threshold'],
        'roc_auc': round(res['roc_auc'], 4),
        'pr_auc': round(res['pr_auc'], 4),
        'best_threshold_metrics': res['best_row'],
        'per_threshold': res['threshold_rows'],
    }

Path(metrics_json_path).parent.mkdir(parents=True, exist_ok=True)
with open(metrics_json_path, 'w') as f:
    json.dump(metrics_output, f, indent=2, default=str)
print(f'Metrics JSON saved -> {metrics_json_path}')

## 12 — Summary

In [ ]:
# ---------------------------------------------------------------------------
# Final summary
# ---------------------------------------------------------------------------
print('=' * 70)
print('RUN COMPLETE — BOOSTING COMPARISON')
print('=' * 70)
print(f'  Metrics JSON:    {metrics_json_path}')
print(f'  Plots directory: {plots_dir}')
if executed_notebook_path:
    print(f'  Notebook:        {executed_notebook_path}')
print()
print(f'  {"Model":<12} {"F1":>8} {"ROC-AUC":>10} {"PR-AUC":>10} {"Time":>8}')
print(f'  {"-"*12} {"-"*8} {"-"*10} {"-"*10} {"-"*8}')
for name, res in results.items():
    t = train_times[name]
    print(f'  {name:<12} {res["best_row"]["f1"]:>8.4f} {res["roc_auc"]:>10.4f} '
          f'{res["pr_auc"]:>10.4f} {t:>7.1f}s')
print()

f1_winner = max(results, key=lambda n: results[n]['best_row']['f1'])
auc_winner = max(results, key=lambda n: results[n]['roc_auc'])
speed_winner = min(train_times, key=train_times.get)

print(f'  Best F1:        {f1_winner}')
print(f'  Best ROC-AUC:   {auc_winner}')
print(f'  Fastest:        {speed_winner}')
print('=' * 70)